<a href="https://colab.research.google.com/github/Roohikhan12/dataviz-exercises-Roohi-Khan/blob/main/lecture07_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [7]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/content/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [8]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [9]:
# Task 1

df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'


ratings_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
heatmap_df = df[df['rating'].isin(ratings_filter)]


heatmap_data = (
    heatmap_df
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

pivot_table = heatmap_data.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)


fig1 = px.imshow(
    pivot_table,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    labels=dict(x='Release Decade', y='Content Rating', color='Title Count'),
    title='TV-MA Dominates Netflix’s Modern Catalogue While PG Ratings Fade After 2000s'
)


max_val = pivot_table.max().max()

for rating in pivot_table.index:
    for decade in pivot_table.columns:
        if pivot_table.loc[rating, decade] == max_val:
            fig1.add_annotation(
                x=decade,
                y=rating,
                text='Highest concentration',
                showarrow=True,
                arrowhead=2,
                font=dict(color='black')
            )

fig1.update_layout(
    title_x=0.5,
    height=500
)

fig1.show()


Netflix’s catalogue is heavily dominated by TV-MA content in the 2010s and 2020s, showing a strong shift toward mature audiences.
In contrast, family-friendly ratings like PG became far less common after the 2000s, highlighting changing viewer preferences and platform strategy.

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [10]:
# Task 2
movies_df = df[df['type'] == 'Movie']


movies_df = movies_df[
    (movies_df['added_year'] >= 2015) &
    (movies_df['added_year'] <= 2022)
]


yearly_additions = (
    movies_df
    .groupby('added_year')
    .size()
    .reset_index(name='count')
    .sort_values('added_year')
)


years = yearly_additions['added_year'].astype(str).tolist()
counts = yearly_additions['count'].tolist()


x_vals = years + ['Total']
y_vals = counts + [sum(counts)]

measure = ['relative'] * len(counts) + ['total']


fig2 = go.Figure(go.Waterfall(
    name='Movies Added',
    orientation='v',
    measure=measure,
    x=x_vals,
    y=y_vals,
    textposition='outside',
    text=[str(v) for v in y_vals],
    connector={"line": {"color": "gray"}},

    increasing={"marker": {"color": "green"}},
    decreasing={"marker": {"color": "red"}},
    totals={"marker": {"color": "blue"}}
))


max_idx = yearly_additions['count'].idxmax()
max_year = str(yearly_additions.loc[max_idx, 'added_year'])
max_count = yearly_additions.loc[max_idx, 'count']


fig2.add_annotation(
    x=max_year,
    y=max_count,
    text='Biggest yearly expansion',
    showarrow=True,
    arrowhead=2
)

fig2.update_layout(
    title='Netflix Movie Library Accelerated Rapidly Before Reaching a Massive Cumulative Total',
    title_x=0.5,
    xaxis_title='Year',
    yaxis_title='Movies Added',
    height=550
)

fig2.show()


Netflix’s movie library experienced its largest yearly expansion during the late 2010s, reflecting aggressive content growth during that period.
The cumulative total shows how rapidly Netflix scaled its movie catalogue between 2015 and 2022 to strengthen its streaming dominance.